# 01 — EDA descriptivo opcional

Este notebook describe el **desarrollo** de S1/S2/S3 y justifica con gráficas cada decisión del preprocesamiento: exclusión fija de canales, referencia de mediana común, banda 40–200 Hz, notch 60/120/180 Hz, Morlet (40 frecuencias, 7 ciclos), decimación a 100 Hz con antialias, recuperación del guante a 25 Hz e interpolación cúbica, retardo de 200 ms, escalado por fold y validación por bloques.

**Reglas**
- Las transformaciones se aplican **en memoria** con `tt_ecog.preprocessing` (la misma implementación que usará el notebook 02) y solo para graficar; no se guardan datos procesados.
- `train_dg` de desarrollo se usa **solo de forma descriptiva**: las gráficas son confirmatorias y no ajustan parámetros, que siguen fijos por protocolo. Las etiquetas de test **nunca** se leen.
- La exclusión de canales vive solo en `configs/experiment_config.yaml` (`channels.excluded`).

**Flujo VS Code + Colab**
1. Edita y haz commit/push en tu máquina local (las salidas del notebook las quita nbstripout al commitear).
2. Selecciona el kernel **Colab** en VS Code y ejecuta las celdas: la primera hace `git pull` en el runtime y muestra el commit, que debe coincidir con tu `git rev-parse HEAD` local.
3. Cada figura se guarda como PNG en `MyDrive/TT_ECoG/results/eda/<run_id>/fig/` junto con tablas y manifiestos, y se muestra aquí mismo; al guardar el notebook local las gráficas quedan en él.

Si el repositorio es privado, autentica Git de forma interactiva; no escribas tokens en celdas ni URLs.

**Este notebook no es requisito para 02 ni 03.** Sus resultados son descriptivos, no seleccionan parámetros ni autorizan el pipeline.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/SaulRodas/BayesianOptimizationNN_ECOG.git'
BRANCH = 'main'
PROJECT = Path('/content/tt-ecog')
PREVIOUS = None
if not PROJECT.exists():
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPOSITORY, str(PROJECT)], check=True)
else:
    remote = subprocess.check_output(['git', '-C', str(PROJECT), 'remote', 'get-url', 'origin'], text=True).strip()
    if remote != REPOSITORY:
        raise RuntimeError('El checkout existente pertenece a otro remoto; revisa /content/tt-ecog.')
    if subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('El clon de Colab tiene cambios locales; no se actualiza automáticamente.')
    PREVIOUS = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
COMMIT = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True).strip()
if 'tt_ecog' in sys.modules and PREVIOUS is not None and PREVIOUS != COMMIT:
    raise RuntimeError('El código cambió después de importarlo: reinicia el runtime y vuelve a ejecutar.')
print('Commit en Colab:', COMMIT)
print('Debe coincidir con `git rev-parse HEAD` local (haz push antes de ejecutar).')

## Entorno
Se instalan solo las dependencias del EDA que falten (`requirements.txt`: incluye MNE y scikit-learn). No se instala PyTorch ni CUDA. Si un módulo cambia después de importarlo, reinicia el runtime.

In [ ]:
env_check = subprocess.run([sys.executable, str(PROJECT / 'scripts/check_environment.py'), '--install-missing'],
                           capture_output=True, text=True)
print(env_check.stdout + env_check.stderr)  # Colab no muestra la salida de subprocesos sin captura.
env_check.check_returncode()
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', str(PROJECT)], check=True)
# El .pth del modo editable solo se procesa al iniciar el intérprete; el kernel activo no lo ve.
SOURCE = PROJECT / 'src'
if str(SOURCE) not in sys.path:
    sys.path.insert(0, str(SOURCE))
import tt_ecog
if not Path(tt_ecog.__file__).resolve().is_relative_to(SOURCE.resolve()):
    raise RuntimeError(f'tt_ecog se importa desde {tt_ecog.__file__}, fuera del checkout; reinicia el runtime.')
print('tt_ecog', tt_ecog.__version__, '->', tt_ecog.__file__)

## Drive y nueva corrida
Puedes usar `Colab: Mount Google Drive to Server...` o ejecutar la celda. Las rutas salen del YAML: datos en `TT_ECoG/raw`, resultados en `TT_ECoG/results/eda/<run_id>/`. Cada ejecución crea una carpeta nueva; nunca se sobrescribe evidencia previa.

In [ ]:
from google.colab import drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.is_dir():
    drive.mount('/content/drive')
CONFIG = PROJECT / 'configs/experiment_config.yaml'


In [ ]:
from tt_ecog.eda import finish_run, load_subjects, show, start_run
ctx = start_run(CONFIG, DRIVE_ROOT)
print('Corrida:', ctx.run_id)
print('Resultados en Drive:', ctx.output)
print('Commit registrado:', ctx.env['git_commit'], '| cambios sin commitear:', ctx.env['git_dirty'])
import numpy as np
import pandas as pd
from tt_ecog import preprocessing as pp

## s01 · Datos de desarrollo
Carga `train_data` y `train_dg` de S1/S2/S3 (solo lectura, con hash antes y después) y muestra integridad, mapa crudo y trazas. Datos ausentes → `BLOCKED`; contrato inválido → `FAIL`.

In [ ]:
subjects = load_subjects(ctx)
for b in subjects.values():
    b.kept = pp.exclude_channels(b.raw, b.cmap)
    b.common = pp.common_median(b.kept)
    b.local_std = b.compute_local_std()
    b.std_ratio = b.compute_std_ratio()
    b.raw_psd = b.compute_raw_psd()
    b.channel_summary = b.compute_channel_summary()
display(pd.DataFrame(ctx.subject_status).T)

In [ ]:
from tt_ecog.eda.sections.s01_overview import draw as plot_overview, TITLE, JUSTIFIES
ctx.begin_section('s01_overview', TITLE, JUSTIFIES)
plot_overview(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s02 · Canales atípicos → exclusión fija
Variabilidad local por canal y en el tiempo, ranking pico/típico, episodios y PSD. Las figuras `s02_S1_c054`, `s02_S2_c020` y `s02_S2_c037` son la evidencia que cita el YAML para excluir esos canales **antes** de la CMR. Las líneas 10×/20×/50× son referencias descriptivas, no umbrales.

In [ ]:
from tt_ecog.eda.sections.s02_channels import draw as plot_channels, TITLE, JUSTIFIES
ctx.begin_section('s02_channels', TITLE, JUSTIFIES)
plot_channels(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s03 · Referencia de mediana común (CMR)
Compara media (CAR) y mediana (CMR), con todos los canales y solo con los conservados, durante los peores episodios; robustez en todo el registro; espectro, fracción de línea y correlación entre canales antes/después de la CMR.

### Cálculo: excluir y referenciar
Se conservan arrays explícitos para comparar antes y después.

In [ ]:
for b in subjects.values():
    b.kept = pp.exclude_channels(b.raw, b.cmap)
    b.cmr = pp.common_median_reference(b.kept)
    b.cmr_psd = b.compute_cmr_psd()
    print(b.subject, b.raw.shape, '→', b.cmr.shape)

### Cálculo: señal filtrada para la comparación espectral
Se usa el FIR combinado; el diseño se examina en las secciones de banda y notch.

In [ ]:
for b in subjects.values():
    b.filtered = pp.apply_fir(b.cmr, b.spec.fir)
    b.filtered_psd = b.compute_filtered_psd()

In [ ]:
from tt_ecog.eda.sections.s03_reference import reference_signals
for b in subjects.values():
    b.reference_signals = reference_signals(b, slice(0, b.n_samples))
    window = b.excerpt(ctx.config.data['eda']['correlation_excerpt_seconds'])
    before = pp.apply_fir(b.kept[:, window], b.spec.fir)
    interior = pp.valid_slice(before.shape[1], b.spec.fir.half_support)
    b.reference_correlation_inputs = (before[:, interior], b.filtered[:, window][:, interior])

In [ ]:
from tt_ecog.eda.sections.s03_reference import draw as plot_reference, TITLE, JUSTIFIES
ctx.begin_section('s03_reference', TITLE, JUSTIFIES)
plot_reference(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s04 · Banda 40–200 Hz
Espectro 1–500 Hz, cambio movimiento/reposo y ERSP alrededor de inicios de movimiento (etiquetas de desarrollo, descriptivo), y diseño del pasabanda.

### Datos derivados en memoria para contrastes descriptivos
Las gráficas de banda usan movimiento y representación alta gamma. Se calculan aquí explícitamente
con parámetros fijos; las secciones 06–09 explican su resolución, remuestreo y alineación.
Nada de estos arrays se guarda ni alimenta al notebook 02.

In [ ]:
for b in subjects.values():
    b._power = pp.morlet_power_decimated(b.filtered, b.spec)
    b.t100 = pp.output_times(b.n_samples, b.spec.factor, b.fs)
    b.edge100 = pp.support_report(b.spec)['total_half_support_samples_target']
    b.hg100 = b.compute_hg100()
    print(b.subject, 'potencia:', b.power100.shape, '| borde:', b.edge100)

In [ ]:
for b in subjects.values():
    b.glove_inspection = pp.glove_storage_inspection(b.glove_record.values,
        b.glove_record.metadata['sampling_hz_glove_storage'], b.spec)
    y25, knots = pp.effective_glove(b.glove_record.values, b.glove_inspection, b.spec)
    b.glove25 = (y25, knots / b.fs)
    b.glove100 = pp.interpolate_glove(b.glove25[1], y25, b.t100)
    b.valid100 = b.glove100[1] & pp.feature_valid_mask(b.n_samples, b.spec)
    b.movement = b.compute_movement()
    b.blocks = b.compute_blocks()
    b.descriptive_channel = b.compute_descriptive_channel()
    print(b.subject, b.glove_inspection['method'], '| flexión:', b.glove100[0].shape)

In [ ]:
from tt_ecog.eda.sections.s04_band import movement_rest_ratio, event_related_power
for b in subjects.values():
    b.movement_rest_spectrum = movement_rest_ratio(b, ctx.config.data['eda'])
    b.event_related_power = event_related_power(b, ctx.config.data['eda'])

In [ ]:
from tt_ecog.eda.sections.s04_band import draw as plot_band, TITLE, JUSTIFIES
ctx.begin_section('s04_band', TITLE, JUSTIFIES)
plot_band(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s05 · Notch 60/120/180 Hz
Espectros de alta resolución alrededor de cada línea, estacionariedad de la línea de 60 Hz, fracción de línea por etapa (S2 muy contaminado) y respuesta del notch.

In [ ]:
from tt_ecog.descriptive import welch_psd
from scipy.signal import spectrogram
for b in subjects.values():
    eda = ctx.config.data['eda']
    b.line_spectra = [welch_psd(data, b.fs, eda['line_zoom_psd_window_seconds'], eda['psd_overlap_fraction'])
                      for data in (b.kept, b.cmr, b.filtered)]
    nper = int(round(eda['psd_window_seconds'] * b.fs))
    b.line_spectrogram = spectrogram(b.kept, fs=b.fs, window='hann', nperseg=nper,
                                     noverlap=nper // 2, detrend='constant', axis=-1)

In [ ]:
from tt_ecog.eda.sections.s05_notch import draw as plot_notch, TITLE, JUSTIFIES
ctx.begin_section('s05_notch', TITLE, JUSTIFIES)
plot_notch(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s06 · Morlet: 40 frecuencias, 7 ciclos, zero_mean
Familia de wavelets y resolución tiempo-frecuencia, escalograma con flexión, efecto de zero_mean y de los bordes (base de las máscaras).

In [ ]:
from tt_ecog.eda.sections.s06_morlet import offset_power, edge_power
for b in subjects.values():
    from mne.time_frequency import morlet
    if b.subject == ctx.config.data['project']['subjects'][0]:
        b.wavelets = morlet(b.fs, b.spec.freqs, n_cycles=b.spec.n_cycles, zero_mean=True)
        b.offset_power = offset_power(b, ctx.config.data['eda'])
        b.edge_power = edge_power(b)

In [ ]:
from tt_ecog.eda.sections.s06_morlet import draw as plot_morlet, TITLE, JUSTIFIES
ctx.begin_section('s06_morlet', TITLE, JUSTIFIES)
plot_morlet(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s07 · Decimación a 100 Hz con antialias
Ancho de banda de la envolvente de potencia, submuestreo directo frente a antialias, coherencia alta gamma–flexión y respuesta del filtro.

In [ ]:
from tt_ecog.eda.sections.s07_decimation import power_envelopes
from tt_ecog.eda import comparators
from scipy.signal import coherence
for b in subjects.values():
    b.power_envelopes = power_envelopes(b, ctx.config.data['eda']['correlation_excerpt_seconds'])
    envelope = b.power_envelopes[0][:, -1]
    b.direct_envelope = comparators.naive_slice(envelope, b.spec.factor)
    b.antialiased_envelope = pp.decimate(envelope, b.spec.antialias, b.spec.factor)
    x, y = b.hg100[:, b.valid100], b.glove100[0][:, b.valid100]
    b.glove_coherence = [coherence(x, y[finger][None, :], fs=b.spec.target_sfreq,
                                 nperseg=min(512, x.shape[1] // 2), axis=-1) for finger in range(5)]

In [ ]:
from tt_ecog.eda.sections.s07_decimation import draw as plot_decimation, TITLE, JUSTIFIES
ctx.begin_section('s07_decimation', TITLE, JUSTIFIES)
plot_decimation(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s08 · Guante: 25 Hz y spline cúbico
Identificación del patrón de almacenamiento a 1 kHz (método y fase), recuperación de las muestras efectivas, espectro del guante y validación de la interpolación con muestras ocultas.

In [ ]:
from tt_ecog.eda.sections.s08_glove import interpolation_holdout
from tt_ecog.eda import comparators
for b in subjects.values():
    b.linear_glove = comparators.linear(b.glove25[1], b.glove25[0], b.t100)
    b.interpolation_holdout = interpolation_holdout(*b.glove25)

In [ ]:
from tt_ecog.eda.sections.s08_glove import draw as plot_glove, TITLE, JUSTIFIES
ctx.begin_section('s08_glove', TITLE, JUSTIFIES)
plot_glove(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s09 · Retardo de 200 ms
Correlación alta gamma–flexión en función del retardo: los picos en retardos positivos (ECoG adelanta) sostienen ECoG(t) → flexión(t+0.2 s).

In [ ]:
from tt_ecog.eda.sections.s09_delay import lagged_correlation
for b in subjects.values():
    lag_range = ctx.config.data['eda']['lag_range_seconds']
    fs = b.spec.target_sfreq
    b.lagged_correlation = lagged_correlation(b, int(round(lag_range[1] * fs)), int(round(lag_range[0] * fs)))

In [ ]:
from tt_ecog.eda.sections.s09_delay import draw as plot_delay, TITLE, JUSTIFIES
ctx.begin_section('s09_delay', TITLE, JUSTIFIES)
plot_delay(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s10 · Escalado por fold
Colas pesadas de la potencia, STD frente a la escala de RobustScaler(10, 90), deriva entre bloques y MinMax ajustado sin el bloque evaluado.

In [ ]:
from tt_ecog.eda.sections.s10_scaling import scale_ratio, minmax_outside
for b in subjects.values():
    b.scale_ratio = scale_ratio(b, ctx.config)
    b.minmax_outside = minmax_outside(b, ctx.config)

In [ ]:
from tt_ecog.eda.sections.s10_scaling import draw as plot_scaling, TITLE, JUSTIFIES
ctx.begin_section('s10_scaling', TITLE, JUSTIFIES)
plot_scaling(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## s11 · Validación por bloques y soporte temporal
Bloques B1–B5 (validación de la etapa 1 sombreada), autocorrelación de flexión y potencia, y soporte medido de la cadena frente al candidato de purga. La purga sigue pendiente hasta el paso 3.

In [ ]:
from tt_ecog.eda.sections.s11_validation import autocorrelation
for b in subjects.values():
    max_lag = int(min(ctx.config.data['eda']['autocorr_max_lag_seconds'] * b.spec.target_sfreq, b.valid100.sum() // 2))
    b.glove_autocorrelation = autocorrelation(b.glove100[0][:, b.valid100], max_lag)
    b.power_autocorrelation = autocorrelation(b.hg100[:, b.valid100], max_lag)
    if b.subject == ctx.config.data['project']['subjects'][0]:
        b.measured_support = pp.measured_support(b.spec, ctx.config.data['verification']['support_tolerance'])

In [ ]:
from tt_ecog.eda.sections.s11_validation import draw as plot_validation, TITLE, JUSTIFIES
ctx.begin_section('s11_validation', TITLE, JUSTIFIES)
plot_validation(ctx, subjects)
show(ctx.end_section())

**Interpretación:** revisar las figuras y tablas de esta sección. Son evidencia descriptiva; no cambian parámetros del YAML ni habilitan otras etapas.

## Cierre
Escribe `run_summary.json`, `index.md` y `artifact_manifest.json` (SHA-256 de cada archivo) en la carpeta de la corrida. Falla si alguna sección no terminó en `COMPLETE`. Guarda el notebook local para conservar las gráficas y anota el `run_id` en `docs/experimental_log.md`.

In [ ]:
summary = finish_run(ctx)
print('Estado:', summary['status'])
print('Carpeta:', ctx.output)